## Open In Colab\n\n[![Run in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/waymax-simulation-experiments/blob/main/experiments/risk-uq-suite/notebooks/10_decision_audit/decision_audit_artifact_builder_harder_colab.ipynb)


# Risk/UQ Track: Decision-Audit Artifact Builder (Harder Preset)
## Objective
Generate a harder closed-loop probe run with more high-interaction/shift-positive mass so downstream decision audits have enough positive support for statistically meaningful conclusions.
## Inputs
- Experiment config from `configs/experiments/risk-uq-suite.json`
- Waymax/LatentDriver runtime bootstrap in Colab
## Outputs (for downstream notebooks)
- `*_miscalibration_probe_predictions.parquet`
- `*_miscalibration_probe_summary.csv`
- `*_miscalibration_probe_threshold_diagnostics.csv`
- `*_miscalibration_probe_reliability_bins.csv`
- `*_miscalibration_probe_class_balance.csv`
- Contract manifest files mirrored to persistent storage
## Downstream Consumers
1. `cross_signal_decision_audit_colab.ipynb`
2. `oracle_bottleneck_colab.ipynb`
3. `planner_practice_method_benchmark_colab.ipynb`
## Harder Preset Intent
- Increase scenario/evaluation sample sizes and holdout pressure for stronger failure-signal coverage.
- Keep artifact contract identical, so existing decision-audit notebooks can read outputs without changes.


## Execution Design
- This notebook is the producer stage for harder decision-audit artifacts.
- It runs in fresh mode on first launch and auto-resumes the same run after Colab restarts.
- Artifacts are written under `PERSIST_ROOT` in Google Drive and later consumed by decision-audit notebooks via `analysis_run_prefix`.
## Reliability Requirements
1. Mount Drive and verify write access before running simulation.
2. Run smoke gates before main execution.
3. Export notebook contract manifests at start and completion.
4. Keep `RUN_ENABLED=True` unless intentionally dry-running setup only.
## Harder Preset Requirements
- Use dedicated run prefix (default: `risk_uq_v2_hard`) to avoid mixing with standard runs.
- Raise scenario/eval sizes and calibration support counts for better tau-sweep confidence.


## Execution Design
- This notebook is the producer stage for decision-audit artifacts.
- It can run in **fresh mode** (default here) so you generate a new run prefix and avoid accidental reuse.
- Artifacts are written under `PERSIST_ROOT` in Google Drive and later discovered by analysis notebooks using `analysis_run_prefix` or auto-discovery.
## Reliability Requirements
1. Mount Drive and verify write access before running simulation.
2. Run smoke gates before main execution.
3. Export notebook contract manifests at start and completion.
4. Keep `RUN_ENABLED=True` unless intentionally dry-running setup only.
- This notebook now persists the selected run tag to Drive and auto-resumes that run after Colab restart.


## Step 1 - Deterministic Bootstrap Constants


In [ ]:
import os
import subprocess
import sys
from pathlib import Path
REPO_URL = 'https://github.com/achyutmorang/waymax-simulation-experiments.git'
REPO_DIR = '/content/waymax-simulation-experiments'
REPO_BRANCH = 'main'
EXPERIMENT_SLUG = 'risk-uq-suite'
EXPERIMENT_CONFIG_PATH = f'{REPO_DIR}/configs/experiments/{EXPERIMENT_SLUG}.json'
REQUIRED_DRIVE_FOLDER = '/content/drive/MyDrive/waymax_experiments'
runtime_cfg_overrides = {
    'verify_drive_access_every_run': False,
    'force_reinstall': False,
    'auto_restart_after_setup': True,
    'strict_lockfile_check': True,
    'setup_cache_enabled': True,
    'revalidate_core_imports_on_cache_hit': True,
    'setup_cache_path': '/content/.closedloop_setup_cache.json',
    'force_module_hot_reload': True,
}
content_root = Path('/content')
content_root.mkdir(parents=True, exist_ok=True)
try:
    _ = os.getcwd()
except FileNotFoundError:
    os.chdir(str(content_root))
NOTEBOOK_NAME = 'decision_audit_artifact_builder_harder_colab'


## Step 2 - Storage Setup


In [ ]:
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception as exc:
    print('[storage] google.colab drive mount unavailable:', exc)
drive_root = Path('/content/drive/MyDrive')
if not drive_root.exists():
    raise RuntimeError('Drive root not found at /content/drive/MyDrive. Ensure Google Drive is mounted.')
required_root = Path(REQUIRED_DRIVE_FOLDER)
required_root.mkdir(parents=True, exist_ok=True)
probe = required_root / '.risk_uq_storage_probe'
probe.write_text('ok')
probe.unlink(missing_ok=True)
print('[storage] ready:', required_root)


## Step 3 - Repo Sync + Runtime Bootstrap


In [ ]:
if not Path(REPO_DIR).exists():
    subprocess.run(['git', 'clone', '--depth', '1', '-b', REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
for path in (REPO_DIR, f'{REPO_DIR}/src'):
    if path not in sys.path:
        sys.path.insert(0, path)
from src.platform import ColabRuntimeConfig, bootstrap_colab_runtime_with_config
runtime_cfg = ColabRuntimeConfig(
    repo_url=REPO_URL,
    repo_dir=REPO_DIR,
    repo_branch=REPO_BRANCH,
    required_drive_folder=REQUIRED_DRIVE_FOLDER,
    **runtime_cfg_overrides,
)
bootstrap = bootstrap_colab_runtime_with_config(runtime_cfg)
globals()['_RISK_UQ_REPO_REV'] = str(bootstrap.repo_sync.repo_rev)
setup_result = bootstrap.setup.result
print('Working directory:', os.getcwd())
print('Repo commit:', bootstrap.repo_sync.repo_rev)
print('Drive mounted:', bootstrap.drive_status.mounted)
print('[setup] result:', setup_result)
if bool(setup_result.get('restart_required', False)) and (not bool(runtime_cfg.auto_restart_after_setup)):
    print('[setup] restart_required=True; restart runtime before continuing.')


## Step 4 - Configuration + Run Context (Fresh Producer Mode)


In [ ]:
from pathlib import Path
import pandas as pd
from src.workflows import (
    initialize_risk_uq_run_context,
    load_experiment_config,
    report_risk_uq_run_context,
)
EXPERIMENT_CFG = load_experiment_config(
    slug=EXPERIMENT_SLUG,
    repo_root=REPO_DIR,
    default_on_missing=False,
)
run_cfg = dict(EXPERIMENT_CFG.get('run', {}))
# Mandatory contract fields
RUN_NAME = str(run_cfg.get('run_name', '')).strip()
RUN_PREFIX = str(run_cfg.get('run_prefix', 'risk_uq_v2')).strip() or 'risk_uq_v2'
PERSIST_ROOT = str(run_cfg.get('persist_root', '/content/drive/MyDrive/waymax_experiments/risk_uq_runs/v2')).strip()
N_SHARDS = int(max(1, int(run_cfg.get('n_shards', 1))))
SHARD_ID = run_cfg.get('shard_id', 'auto')
RESUME_FROM_EXISTING = bool(run_cfg.get('resume_from_existing', True))
RUN_ENABLED = bool(run_cfg.get('run_enabled', True))
# Harder preset uses a dedicated prefix by default to prevent artifact mixing.
HARDER_USE_DEDICATED_PREFIX = bool(run_cfg.get('decision_audit_harder_use_dedicated_prefix', True))
if HARDER_USE_DEDICATED_PREFIX:
    RUN_PREFIX = str(run_cfg.get('decision_audit_harder_run_prefix', f'{RUN_PREFIX}_hard')).strip() or f'{RUN_PREFIX}_hard'
RUN_TAG_PREFIX = str(run_cfg.get('run_tag_prefix', RUN_PREFIX)).strip() or RUN_PREFIX
if HARDER_USE_DEDICATED_PREFIX:
    RUN_TAG_PREFIX = str(run_cfg.get('decision_audit_harder_run_tag_prefix', RUN_PREFIX)).strip() or RUN_PREFIX
RUN_MODE_CFG = str(run_cfg.get('run_mode', 'auto')).strip().lower() or 'auto'
RUN_MODE = RUN_MODE_CFG if RESUME_FROM_EXISTING else 'fresh'
RUN_TAG = RUN_NAME
# Producer behavior:
# - First launch: force fresh run (default)
# - Later launches (after runtime restarts): auto-resume same run from sentinel file in Drive
FORCE_FRESH_PROBE_BUILD = bool(run_cfg.get('decision_audit_harder_force_fresh_probe', run_cfg.get('decision_audit_force_fresh_probe', True)))
AUTO_RESUME_AFTER_RESTART = bool(run_cfg.get('decision_audit_harder_auto_resume_after_restart', run_cfg.get('decision_audit_auto_resume_after_restart', True)))
SENTINEL_DIR = Path(PERSIST_ROOT) / RUN_PREFIX
SENTINEL_PATH = SENTINEL_DIR / '_decision_audit_artifact_builder_harder_last_run.txt'
SAVED_RUN_TAG = ''
if AUTO_RESUME_AFTER_RESTART and SENTINEL_PATH.exists():
    try:
        SAVED_RUN_TAG = SENTINEL_PATH.read_text(encoding='utf-8').strip()
    except Exception:
        SAVED_RUN_TAG = ''
if AUTO_RESUME_AFTER_RESTART and SAVED_RUN_TAG:
    RUN_NAME = SAVED_RUN_TAG
    RUN_TAG = SAVED_RUN_TAG
    RESUME_FROM_EXISTING = True
    RUN_MODE = 'resume'
    FORCE_FRESH_PROBE_BUILD = False
elif FORCE_FRESH_PROBE_BUILD:
    RESUME_FROM_EXISTING = False
    RUN_MODE = 'fresh'
    RUN_NAME = ''
    RUN_TAG = ''
run_ctx = initialize_risk_uq_run_context(
    run_tag=RUN_TAG,
    run_tag_prefix=RUN_TAG_PREFIX,
    persist_root=PERSIST_ROOT,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    resume_mode=RUN_MODE,
    resume_from_existing=bool(RESUME_FROM_EXISTING),
    planner_backend='latentdriver',
)
cfg = run_ctx.cfg
search_cfg = {}
RUN_TAG = str(run_ctx.run_tag)
RUN_NAME = str(RUN_NAME or RUN_TAG)
SHARD_ID = int(run_ctx.shard_id)
run_prefix = run_ctx.run_prefix
# Persist selected run tag immediately so a runtime restart can resume this exact run.
if AUTO_RESUME_AFTER_RESTART:
    try:
        SENTINEL_DIR.mkdir(parents=True, exist_ok=True)
        SENTINEL_PATH.write_text(str(RUN_TAG), encoding='utf-8')
    except Exception as exc:
        print('[resume sentinel] warning:', exc)
print('EXPERIMENT_CONFIG_PATH =', EXPERIMENT_CONFIG_PATH)
print('run_prefix (flat artifacts) =', run_prefix)
print('RUN_PREFIX/RUN_NAME (contract dir) =', RUN_PREFIX, RUN_NAME)
print('RUN_ENABLED =', RUN_ENABLED, ' RESUME_FROM_EXISTING =', RESUME_FROM_EXISTING)
print('FORCE_FRESH_PROBE_BUILD =', FORCE_FRESH_PROBE_BUILD)
print('AUTO_RESUME_AFTER_RESTART =', AUTO_RESUME_AFTER_RESTART)
print('SENTINEL_PATH =', SENTINEL_PATH)
report_risk_uq_run_context(run_ctx, display_fn=display)
# Probe knobs
cfg.uq_eval_probability_bins = int(run_cfg.get('decision_audit_harder_uq_eval_probability_bins', 15))
cfg.risk_control_fail_budget = float(run_cfg.get('decision_audit_harder_risk_budget_tau', 0.20))
cfg.uq_shift_suites = tuple(run_cfg.get('decision_audit_harder_shift_suites', (
    'nominal_clean',
    'hist_prim_shift',
    'fut_prim_shift',
    'hist_rmv_shift',
    'high_interaction_holdout',
)))
# Harder-run preset for stronger positive/support mass.
HARDER_PRESET_ENABLE = bool(run_cfg.get('decision_audit_harder_preset_enable', True))
if HARDER_PRESET_ENABLE:
    cfg.n_total_scenarios = int(run_cfg.get('decision_audit_harder_n_total_scenarios', 12000))
    cfg.n_eval_scenarios = int(run_cfg.get('decision_audit_harder_n_eval_scenarios', 1200))
    cfg.strict_min_eval = int(run_cfg.get('decision_audit_harder_strict_min_eval', 1000))
    cfg.high_quantile = float(run_cfg.get('decision_audit_harder_high_quantile', 0.75))
    cfg.n_closedloop_calib = int(run_cfg.get('decision_audit_harder_n_closedloop_calib', 240))
    cfg.n_surprise_calib_proposals = int(run_cfg.get('decision_audit_harder_n_surprise_calib_proposals', 8))
harder_summary = pd.DataFrame([
    {'field': 'HARDER_PRESET_ENABLE', 'value': int(HARDER_PRESET_ENABLE)},
    {'field': 'n_total_scenarios', 'value': int(cfg.n_total_scenarios)},
    {'field': 'n_eval_scenarios', 'value': int(cfg.n_eval_scenarios)},
    {'field': 'strict_min_eval', 'value': int(cfg.strict_min_eval)},
    {'field': 'high_quantile', 'value': float(cfg.high_quantile)},
    {'field': 'n_closedloop_calib', 'value': int(cfg.n_closedloop_calib)},
    {'field': 'n_surprise_calib_proposals', 'value': int(cfg.n_surprise_calib_proposals)},
    {'field': 'risk_control_fail_budget', 'value': float(cfg.risk_control_fail_budget)},
    {'field': 'uq_shift_suites', 'value': ','.join(cfg.uq_shift_suites)},
])
display(harder_summary)
cfg


## Step 5 - Closed-Loop Context + Smoke Gates


In [ ]:
import numpy as np
import pandas as pd
from src.workflows import (
    build_risk_uq_simulation_context,
    load_existing_risk_dataset_artifact,
    run_risk_uq_smoke_gates,
)
existing_dataset_df = load_existing_risk_dataset_artifact(cfg.run_prefix)
if RUN_MODE == 'fresh':
    existing_dataset_df = pd.DataFrame()
needs_simulation_context = bool(existing_dataset_df.empty)
runner = None
eval_idx = None
print('existing_dataset_rows =', len(existing_dataset_df))
print('needs_simulation_context =', needs_simulation_context)
if needs_simulation_context:
    sim_bundle = build_risk_uq_simulation_context(cfg=cfg, n_shards=N_SHARDS, shard_id=SHARD_ID)
    runner = sim_bundle.runner
    eval_idx = sim_bundle.eval_idx
    gate_bundle = run_risk_uq_smoke_gates(
        runner=runner,
        cfg=cfg,
        eval_idx=eval_idx,
        probe_shift_suite='nominal_clean',
    )
    risk_probe_pass = bool(gate_bundle.get('risk_probe_pass', False))
    preflight_pass = bool(gate_bundle.get('preflight_pass', False))
    overall_pass = bool(gate_bundle.get('overall_pass', False))
    display(gate_bundle.get('risk_probe_summary_df', pd.DataFrame()))
    if isinstance(gate_bundle.get('preflight_df', None), pd.DataFrame):
        display(gate_bundle.get('preflight_df', pd.DataFrame()))
else:
    gate_bundle = {'failure_reasons': []}
    required_cols = [
        'scenario_id',
        'dist_top1_weight',
        'dist_entropy',
        'dist_num_components',
        'failure_proxy_h15',
    ]
    required_ok = all(col in existing_dataset_df.columns for col in required_cols)
    finite_ok = False
    if required_ok and len(existing_dataset_df) > 0:
        sample = existing_dataset_df.loc[:, ['dist_top1_weight', 'dist_entropy', 'dist_num_components', 'failure_proxy_h15']].head(4096)
        finite_ok = bool(np.isfinite(sample.to_numpy(dtype=float)).all())
    risk_probe_pass = bool((len(existing_dataset_df) > 0) and required_ok and finite_ok)
    preflight_pass = bool(risk_probe_pass)
    overall_pass = bool(risk_probe_pass and preflight_pass)
print('risk_probe_pass =', risk_probe_pass)
print('preflight_pass =', preflight_pass)
print('overall_pass =', overall_pass)
if not overall_pass:
    raise RuntimeError(f"Miscalibration probe gates failed: {gate_bundle.get('failure_reasons', [])}")


## Step 6 - Manifest + Contract Layout (Gates Passed)


In [ ]:
from src.workflows import write_contract_storage_mirror, write_notebook_contract_manifest
manifest_path = write_notebook_contract_manifest(
    run_prefix=cfg.run_prefix,
    run_tag=RUN_TAG,
    cfg=cfg,
    search_cfg=search_cfg,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    notebook_name=NOTEBOOK_NAME,
    stage='decision_audit_artifact_build_harder_started',
    repo_dir=REPO_DIR,
    git_commit=globals().get('_RISK_UQ_REPO_REV', ''),
    quick_probe_pass=bool(risk_probe_pass),
    preflight_pass=bool(preflight_pass),
)
contract_paths = write_contract_storage_mirror(
    persist_root=PERSIST_ROOT,
    run_prefix=RUN_PREFIX,
    run_name=RUN_NAME,
    run_prefix_path=cfg.run_prefix,
    cfg=cfg,
    search_cfg=search_cfg,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    stage='decision_audit_artifact_build_harder_started',
    git_commit=str(globals().get('_RISK_UQ_REPO_REV', '')),
    resume_from_existing=bool(RESUME_FROM_EXISTING),
    run_enabled=bool(RUN_ENABLED),
)
print('manifest_path =', manifest_path)
print('contract_run_dir =', contract_paths['contract_run_dir'])


## Step 7 - Main Execution (Generate Probe Artifacts)


In [ ]:
from src.workflows import run_miscalibration_probe_flow
# Run exact probe flow used by decision-audit notebooks.
probe_bundle = None
if not bool(RUN_ENABLED):
    print('[main] skipped: RUN_ENABLED=False')
else:
    probe_bundle = run_miscalibration_probe_flow(
        cfg=cfg,
        dataset_df=existing_dataset_df if not existing_dataset_df.empty else None,
        runner=runner,
        eval_idx=eval_idx,
        run_prefix=cfg.run_prefix,
        resume_mode=RUN_MODE,
    )
    print('loaded_from_existing =', probe_bundle.loaded_from_existing)
    display(probe_bundle.benchmark_bundle.summary_df)


## Step 8 - Evaluation/Diagnostics


In [ ]:
if probe_bundle is None:
    print('[report] no run output because RUN_ENABLED=False')
else:
    display(probe_bundle.benchmark_bundle.per_shift_df.head(30))
    display(probe_bundle.benchmark_bundle.reliability_df.head(30))
    display(probe_bundle.threshold_df.head(30))
    if hasattr(probe_bundle, 'proxy_calibration_df'):
        display(probe_bundle.proxy_calibration_df.head(20))
    if hasattr(probe_bundle, 'leakage_df'):
        display(probe_bundle.leakage_df.head(20))
    if hasattr(probe_bundle, 'shift_profile_df'):
        display(probe_bundle.shift_profile_df.head(30))
    if hasattr(probe_bundle, 'class_balance_df'):
        display(probe_bundle.class_balance_df.head(30))
    key = probe_bundle.threshold_df.copy()
    if (not key.empty) and ('variant' in key.columns):
        cols = ['shift_suite', 'variant', 'empirical_failure_given_accepted', 'threshold', 'budget_violated']
        cols = [c for c in cols if c in key.columns]
        print('[diagnostic] budget check by shift/variant')
        display(key.loc[:, cols].head(30))


## Step 9 - Export + Manifest (Completed)


In [ ]:
if probe_bundle is None:
    stage_name = 'decision_audit_artifact_build_harder_skipped'
    artifact_paths = {}
    metrics_path = None
    extra = {'run_skipped': 1}
else:
    stage_name = 'decision_audit_artifact_build_harder_completed'
    artifact_paths = dict(probe_bundle.artifact_paths)
    metrics_path = str(artifact_paths.get('miscalibration_probe_summary', '')) or None
    extra = {
        'loaded_from_existing': int(bool(probe_bundle.loaded_from_existing)),
        'force_fresh_probe_build': int(bool(globals().get('FORCE_FRESH_PROBE_BUILD', False))),
        'auto_resume_after_restart': int(bool(globals().get('AUTO_RESUME_AFTER_RESTART', False))),
        'resume_sentinel_path': str(globals().get('SENTINEL_PATH', '')),
        'resume_saved_run_tag': str(globals().get('RUN_TAG', '')),
        'harder_preset_enabled': int(bool(globals().get('HARDER_PRESET_ENABLE', False))),
        'summary_rows': int(len(probe_bundle.benchmark_bundle.summary_df)),
        'per_shift_rows': int(len(probe_bundle.benchmark_bundle.per_shift_df)),
        'threshold_rows': int(len(probe_bundle.threshold_df)),
    }
manifest_path = write_notebook_contract_manifest(
    run_prefix=cfg.run_prefix,
    run_tag=RUN_TAG,
    cfg=cfg,
    search_cfg=search_cfg,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    notebook_name=NOTEBOOK_NAME,
    stage=stage_name,
    repo_dir=REPO_DIR,
    git_commit=globals().get('_RISK_UQ_REPO_REV', ''),
    extra_fields=extra,
)
contract_paths = write_contract_storage_mirror(
    persist_root=PERSIST_ROOT,
    run_prefix=RUN_PREFIX,
    run_name=RUN_NAME,
    run_prefix_path=cfg.run_prefix,
    cfg=cfg,
    search_cfg=search_cfg,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    stage=stage_name,
    git_commit=str(globals().get('_RISK_UQ_REPO_REV', '')),
    resume_from_existing=bool(RESUME_FROM_EXISTING),
    run_enabled=bool(RUN_ENABLED),
    artifact_paths=artifact_paths,
    metrics_csv_path=metrics_path,
    extra_fields=extra,
)
print('manifest_path =', manifest_path)
print('contract_run_manifest =', contract_paths['contract_run_manifest'])
